# NB_Silver_Customers — Bronze to Silver Transformation (Upgraded v2 — Educational & Production-Ready)

## Section 1 — Purpose, Business Context & Engineering Decisions
- **Source Table:** `Cloud_Retail_Data_Lakehouse_v2.dbo.customers` → ADLS Bronze Parquet exposed via Fabric Shortcut (`Files/bronze/crm/customers/`).
- **Grain:** One row per order-scoped customer record (`customer_id`, `99,441` rows), while `customer_unique_id` (`96,096` distinct values) identifies the underlying person/account across repeat orders.
- **Target Silver Table:** `silver.customers` (managed Delta table registered in the Fabric Lakehouse Catalog).
- **Quarantine Table:** `silver._quarantine_customers` (stores any row-level entity/key/ZIP/state failures for auditability).
- **Execution Environment:** Built for **Microsoft Fabric (`synapse_pyspark` kernel)** where `spark` is pre-configured automatically. Do **not** add `%pip install pyspark`.

### Key Data Quality Findings Driving This Notebook (`CONFIRMED` on `99,441` source rows)
1. **Clean Keys & Zero Silent Loss (`0` Quarantine expected):** All `99,441` `customer_id` and `96,096` distinct `customer_unique_id` values are valid 32-char lowercase hex strings; all `99,441` `customer_zip_code_prefix` values are 5 digits; all `27` `customer_state` codes match Brazil's official UF list with **`0` state conflicts** against `silver.geolocation_zip_reference`.
2. **Clean Cities (`4,119` distinct values):** Unlike `sellers`, `customers` has **`0` emails, `0` digits-only values, `0` double spaces, `0` non-ASCII accents, and `0` bare state codes (`sp`/`rj`)**. The `22` hyphenated/apostrophe cities (`mogi-guacu`, `embu-guacu`, `ji-parana`, `pingo-d'agua`) are legitimate Brazilian municipality names.
3. **Why We Fix `Transformation_Test.ipynb` Anti-Patterns Here:**
   - Do **not** use `df.dropDuplicates()` before checking PK uniqueness (hides source duplicates).
   - Do **not** use `lpad(..., 5, "0")` before validating 5-digit ZIPs (masks truncated ZIPs like `'123'`).
   - Do **not** silently `.filter(...)` out invalid rows — route them to `silver._quarantine_customers` with `dq_reject_reason`.
   - Use **relative Fabric shortcut paths** (`Files/bronze/...`) instead of `spark.table("bronze.customers")` (Bronze is Parquet files, not catalog tables).
4. **Gold Cross-Table Note (`LEFT JOIN` Required):** `278` customer rows (`157` distinct ZIPs) do not exist in `silver.geolocation_zip_reference` (`19,015` ZIPs). Always use `LEFT JOIN` in Gold `dim_customer` so those `278` customers and their orders are never dropped.

> **WARNING:** **DO NOT EXECUTE THE WRITE STEP UNTIL THE TRANSFORMATION HAS BEEN REVIEWED AND APPROVED.** (`WRITE_ENABLED = False` by default.)

## Section 2 — Configuration & Parameters
### What does this cell do?
- **`WRITE_ENABLED = False`:** Safety lock so you can run the entire notebook as a **Dry Run** first without modifying any tables.
- **Relative Fabric Shortcut Path (`Files/bronze/crm/customers/...`):** Targets the exact Bronze snapshot file to avoid mixing multiple snapshots.
- **`SILVER_SCHEMA = "silver"`:** Ensures `CREATE SCHEMA IF NOT EXISTS silver` runs safely before `.saveAsTable(...)`.
- **`VALID_BR_STATES`:** Strict whitelist of Brazil's 27 official UF state codes.

In [ ]:
from pyspark.sql import functions as F

# =============================================================================
# 1. SAFETY LOCK — Run once as Dry Run (False), then set True to persist Delta
# =============================================================================
WRITE_ENABLED = False

# =============================================================================
# 2. PIPELINE PARAMETERS (Relative Fabric Shortcut Path)
# =============================================================================
BRONZE_DIR = "Files/bronze/crm/customers"
SOURCE_FILE_NAME = "customers_20261121_013117_b0e93f82-84a0-4668-9a8c-9ae1da7dd183.parquet"
BRONZE_PATH = f"{BRONZE_DIR}/{SOURCE_FILE_NAME}"

INGESTION_RUN_ID = "b0e93f82-84a0-4668-9a8c-9ae1da7dd183"
SOURCE_SYSTEM = "sqlserver_olist"
SOURCE_TABLE = "dbo.customers"

SILVER_SCHEMA = "silver"
SILVER_TARGET_TABLE = f"{SILVER_SCHEMA}.customers"
QUARANTINE_TARGET_TABLE = f"{SILVER_SCHEMA}._quarantine_customers"

VALID_BR_STATES = [
    "AC", "AL", "AP", "AM", "BA", "CE", "DF", "ES", "GO",
    "MA", "MT", "MS", "MG", "PA", "PB", "PR", "PE", "PI",
    "RJ", "RN", "RS", "RO", "RR", "SC", "SP", "SE", "TO",
]

print(f"Configured BRONZE_PATH       : {BRONZE_PATH}")
print(f"Configured SILVER_TARGET     : {SILVER_TARGET_TABLE}")
print(f"Configured QUARANTINE_TARGET : {QUARANTINE_TARGET_TABLE}")
print(f"WRITE_ENABLED                : {WRITE_ENABLED}")

## Section 3 — Read Bronze Parquet & Capture Source File Lineage
### What does this cell do?
- Loads the exact Bronze Parquet snapshot for `customers` (`99,441` rows).
- Adds `_source_file` via `F.input_file_name()` for row-level lineage.
- Caches `df_bronze` in memory to speed up schema checks, PK checks, and reconciliation.

In [ ]:
df_bronze = (
    spark.read
    .format("parquet")
    .load(BRONZE_PATH)
    .withColumn("_source_file", F.input_file_name())
)

df_bronze.cache()
bronze_count = df_bronze.count()

print(f"Bronze Customer Row Count: {bronze_count:,}")
df_bronze.printSchema()

## Section 4 — Stage 1: Schema Validation (Fail-Fast Contract Check)
### What does this cell do?
- Verifies that all 5 required source columns (`customer_id`, `customer_unique_id`, `customer_zip_code_prefix`, `customer_city`, `customer_state`) exist before running any transformations.
- Raises a `ValueError` immediately if any column is missing (`Schema Drift`).

In [ ]:
REQUIRED_COLUMNS = [
    "customer_id",
    "customer_unique_id",
    "customer_zip_code_prefix",
    "customer_city",
    "customer_state",
]

missing_columns = [c for c in REQUIRED_COLUMNS if c not in df_bronze.columns]
if missing_columns:
    raise ValueError(f"Schema validation failed — missing required columns: {missing_columns}")

print("Stage 1 Schema Validation: PASSED")

## Section 5 — Stage 2: Text & Type Standardization (Without Hiding Data Errors)
### What does this cell do?
- Normalizes `customer_id` and `customer_unique_id` to trimmed lowercase strings so downstream joins with `orders` match deterministically.
- Trims `customer_zip_code_prefix` as a `string` **without** blind `lpad(..., 5, "0")` before validation, so malformed short ZIPs cannot accidentally pass the 5-digit check.
- Standardizes `customer_city` (`lower`, `trim`, collapse internal multi-spaces `r"\s+"`) and `customer_state` (`upper`, `trim`).

In [ ]:
df_std = (
    df_bronze
    .withColumn("customer_id", F.lower(F.trim(F.col("customer_id").cast("string"))))
    .withColumn("customer_unique_id", F.lower(F.trim(F.col("customer_unique_id").cast("string"))))
    .withColumn("customer_zip_code_prefix", F.trim(F.col("customer_zip_code_prefix").cast("string")))
    .withColumn("customer_city", F.lower(F.trim(F.regexp_replace(F.col("customer_city").cast("string"), r"\s+", " "))))
    .withColumn("customer_state", F.upper(F.trim(F.col("customer_state").cast("string"))))
)

print("Stage 2 Standardization complete. Distinct cities:", df_std.select("customer_city").distinct().count())

## Section 6 — Stage 3: Primary Key Uniqueness Check (BEFORE Any Filtering — NO `dropDuplicates`)
### Why check PK uniqueness here?
- `customer_id` is the Primary Key (`1 row per order-scoped customer record`), and must match `orders.customer_id` 1:1.
- Calling `dropDuplicates()` first (as `Transformation_Test.ipynb` did) would mask duplicate PK issues in the source instead of surfacing them.

In [ ]:
duplicate_customer_ids = (
    df_std
    .filter(F.col("customer_id").isNotNull() & (F.col("customer_id") != ""))
    .groupBy("customer_id")
    .count()
    .filter(F.col("count") > 1)
)

duplicate_pk_count = duplicate_customer_ids.count()
if duplicate_pk_count > 0:
    raise ValueError(
        f"Primary Key Violation: Found {duplicate_pk_count} duplicate customer_id values. "
        "Stopping without blind dropDuplicates()."
    )

print("Stage 3 Primary Key Uniqueness Check: PASSED (0 duplicate customer_id)")

## Section 7 — Stage 4: Row-Level Entity Validation & Quarantine Split
### What does this cell do?
- Validates every row against 4 strict rules:
  1. `customer_id` must be a valid 32-char lowercase hex string (`^[0-9a-f]{32}$`).
  2. `customer_unique_id` must be a valid 32-char lowercase hex string (`^[0-9a-f]{32}$`).
  3. `customer_zip_code_prefix` must be exactly 5 digits (`^\d{5}$`).
  4. `customer_state` must belong to Brazil's 27 official UF codes (`VALID_BR_STATES`).
- Combines all failure reasons per row using `F.concat_ws(";", ...)` into `dq_reject_reason`, routing any failing rows to `df_quarantine` (`0` expected on current data) instead of dropping them silently.

In [ ]:
customer_id_invalid = F.col("customer_id").isNull() | (~F.col("customer_id").rlike(r"^[0-9a-f]{32}$"))
unique_id_invalid   = F.col("customer_unique_id").isNull() | (~F.col("customer_unique_id").rlike(r"^[0-9a-f]{32}$"))
zip_invalid         = F.col("customer_zip_code_prefix").isNull() | (~F.col("customer_zip_code_prefix").rlike(r"^\d{5}$"))
state_invalid       = F.col("customer_state").isNull() | (~F.col("customer_state").isin(*VALID_BR_STATES))

df_classified = df_std.withColumn(
    "dq_reject_reason",
    F.concat_ws(
        ";",
        F.when(customer_id_invalid, F.lit("INVALID_CUSTOMER_ID")),
        F.when(unique_id_invalid, F.lit("INVALID_CUSTOMER_UNIQUE_ID")),
        F.when(zip_invalid, F.lit("INVALID_ZIP_PREFIX")),
        F.when(state_invalid, F.lit("INVALID_BR_STATE")),
    )
)

df_quarantine = (
    df_classified
    .filter(F.col("dq_reject_reason") != "")
    .withColumn("source_system", F.lit(SOURCE_SYSTEM))
    .withColumn("source_table", F.lit(SOURCE_TABLE))
    .withColumn("ingestion_run_id", F.lit(INGESTION_RUN_ID))
    .withColumnRenamed("_source_file", "source_file")
    .withColumn("quarantined_at", F.current_timestamp())
)

df_valid_entities = df_classified.filter(F.col("dq_reject_reason") == "").drop("dq_reject_reason")

valid_count = df_valid_entities.count()
quarantine_count = df_quarantine.count()

print(f"Valid Customer Entities : {valid_count:,} (Expected: 99,441)")
print(f"Quarantined Rows        : {quarantine_count:,} (Expected: 0)")

## Section 8 — Stage 5: City Quality & Repeat-Buyer Diagnostics
### What does this cell do?
- Performs a non-destructive audit on `customer_city` (checks for nulls, emails, or digits-only strings — **`0` expected**).
- Reports distinct `customer_unique_id` count (`96,096` expected) and repeat-buyer accounts (`2,997` expected) to confirm grain behavior for downstream Gold `dim_customer`.

In [ ]:
EMAIL_REGEX = r"(?i)^[^@\s]+@[^@\s]+\.[^@\s]+$"
DIGITS_ONLY_REGEX = r"^\d+$"

corrupted_city_count = df_valid_entities.filter(
    F.col("customer_city").isNull() |
    (F.col("customer_city") == "") |
    F.col("customer_city").rlike(EMAIL_REGEX) |
    F.col("customer_city").rlike(DIGITS_ONLY_REGEX)
).count()

distinct_unique_customers = df_valid_entities.select("customer_unique_id").distinct().count()
repeat_unique_customers = (
    df_valid_entities.groupBy("customer_unique_id").count().filter(F.col("count") > 1).count()
)

print(f"Corrupted Customer Cities (Null/Email/Digits): {corrupted_city_count} (Expected: 0)")
print(f"Distinct customer_unique_id                  : {distinct_unique_customers:,} (Expected: 96,096)")
print(f"Repeat-Buyer Accounts (>1 order)             : {repeat_unique_customers:,} (Expected: 2,997)")

## Section 9 — Stage 6: Build Final Silver DataFrame & Pre-Write Reconciliation
### What does this cell do?
- Attaches full audit & lineage columns (`source_system`, `source_table`, `source_file`, `ingestion_run_id`, `silver_processed_at`) in a clean 10-column schema.
- Enforces the strict reconciliation rule: `bronze_count == silver_count + quarantine_count` (`99,441 == 99,441 + 0`).

In [ ]:
df_silver = (
    df_valid_entities
    .withColumn("source_system", F.lit(SOURCE_SYSTEM))
    .withColumn("source_table", F.lit(SOURCE_TABLE))
    .withColumn("ingestion_run_id", F.lit(INGESTION_RUN_ID))
    .withColumnRenamed("_source_file", "source_file")
    .withColumn("silver_processed_at", F.current_timestamp())
    .select(
        "customer_id",
        "customer_unique_id",
        "customer_zip_code_prefix",
        "customer_city",
        "customer_state",
        "source_system",
        "source_table",
        "source_file",
        "ingestion_run_id",
        "silver_processed_at",
    )
)

silver_count = df_silver.count()
if bronze_count != silver_count + quarantine_count:
    raise ValueError(
        f"Reconciliation FAILED: bronze({bronze_count}) != silver({silver_count}) + quarantine({quarantine_count})"
    )

df_silver.printSchema()
print(f"Stage 6 Reconciliation PASSED: bronze={bronze_count:,}, silver={silver_count:,}, quarantine={quarantine_count:,}")

## Section 10 — Stage 7: Proposed Silver & Quarantine Write (DISABLED BY DEFAULT)
> **DO NOT EXECUTE THE WRITE STEP UNTIL THE TRANSFORMATION HAS BEEN REVIEWED AND APPROVED.**

### What does this cell do?
- When `WRITE_ENABLED = True`:
  - Runs `CREATE SCHEMA IF NOT EXISTS silver` to avoid `SCHEMA_NOT_FOUND`.
  - Writes `silver.customers` (`99,441` rows) as a managed Delta Catalog table.
  - Appends any quarantined rows to `silver._quarantine_customers` only if `quarantine_count > 0`.

In [ ]:
if WRITE_ENABLED:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER_SCHEMA}")

    (
        df_silver.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(SILVER_TARGET_TABLE)
    )
    print(f"Successfully wrote {silver_count:,} rows to {SILVER_TARGET_TABLE}")

    if quarantine_count > 0:
        (
            df_quarantine.write
            .format("delta")
            .mode("append")
            .option("mergeSchema", "true")
            .saveAsTable(QUARANTINE_TARGET_TABLE)
        )
        print(f"Appended {quarantine_count:,} rows to {QUARANTINE_TARGET_TABLE}")
else:
    print("WRITE_ENABLED is False — dry validation only; no Silver or Quarantine data was written.")

## Section 11 — Post-Write Catalog Validation & Cache Cleanup
### What does this cell do?
- Reads back `spark.table("silver.customers")` after `WRITE_ENABLED = True` to confirm:
  - Total row count (`99,441`).
  - Zero duplicate `customer_id` keys.
  - Exact `96,096` distinct `customer_unique_id` values.
- Unpersists `df_bronze` to free executor memory.

In [ ]:
if WRITE_ENABLED:
    df_check = spark.table(SILVER_TARGET_TABLE)
    final_count = df_check.count()
    if final_count != silver_count:
        raise ValueError(f"Post-write count mismatch: expected {silver_count}, actual {final_count}")

    final_dups = df_check.groupBy("customer_id").count().filter(F.col("count") > 1).count()
    if final_dups > 0:
        raise ValueError(f"Post-write PK violation: {final_dups} duplicate customer_id found")

    final_unique_cust = df_check.select("customer_unique_id").distinct().count()
    print(f"POST-WRITE CATALOG VALIDATION PASSED: {final_count:,} rows, 0 duplicate customer_id, {final_unique_cust:,} unique customers.")
    df_check.printSchema()
else:
    print("Post-write validation skipped because WRITE_ENABLED is False.")

df_bronze.unpersist()